# Precompute 01 — embed the candidate pool on a Colab GPU

This runs the **same** `src/precompute/01_embed_candidates.py` as locally, but on a
GPU, so the full 100k pool takes ~10 min instead of ~14 h on CPU.

**This is offline precompute — allowed to use GPU + network.** `rank.py` never does
any of this; it only loads the artifacts produced here. The output is three files
(`candidate_embeddings.npy`, `candidate_ids.npy`, `embeddings_meta.json`) that you
download into the repo's `artifacts/` and commit.

**Before you start:** `Runtime → Change runtime type → GPU` (a free T4 is plenty).

**Two manual prerequisites:**
1. Push the latest `redrob-ranker` code to GitHub (so the clone below is current).
2. Put `candidates.jsonl` (487 MB) on your Google Drive — it is gitignored, so it
   is **not** in the clone. Default expected path: `MyDrive/candidates.jsonl`.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

## 2. Install sentence-transformers

Use Colab's preinstalled **CUDA** torch — do **not** `pip install -r requirements.txt`
here, because that pins the CPU-only torch build (that file is for the `rank.py`
reproduce path, not for GPU precompute).

In [ ]:
!pip install -q sentence-transformers

## 3. Get the code

Clones the repo if it is not already present, then `cd` into it. If you opened this
notebook from a runtime that already has the repo, just `%cd` into it and skip the clone.

In [ ]:
import os

REPO_URL = "https://github.com/CoffeeAurCode/redrob-ranker.git"
if not os.path.isdir("redrob-ranker"):
    !git clone $REPO_URL
%cd redrob-ranker

## 4. Mount Drive and place the candidate pool

Edit `DATA_SRC` if your file lives elsewhere on Drive. We symlink it into `data/`
where the script expects it; `wc -l` must print **100000**.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DATA_SRC = "/content/drive/MyDrive/candidates.jsonl"  # <-- change if needed
assert os.path.exists(DATA_SRC), f"not found on Drive: {DATA_SRC}"

os.makedirs("data", exist_ok=True)
!ln -sf "$DATA_SRC" data/candidates.jsonl
!wc -l data/candidates.jsonl

## 5. Embed the pool (GPU auto-detected)

`SentenceTransformer` picks CUDA automatically — no code change needed. The script
checkpoints under `artifacts/.embed_ckpt/`, so if the Colab session drops, just re-run
this cell and it resumes from the last completed chunk.

In [ ]:
!python src/precompute/01_embed_candidates.py

## 6. Sanity check — does the ranking look sane?

Embeds a hand-written "ideal Senior AI Engineer" query and prints the top 30. You want
ML / retrieval / ranking / recsys / search / data-science titles near the top — **not** a
keyword-stuffed `Marketing Manager`. If a stuffer dominates, the embedding text is
leaking skills (it should not — `profile_text` excludes them).

In [ ]:
!python src/precompute/sanity_rank.py --top 30

## 7. (Optional) Run the artifact invariant tests

In [ ]:
!pip install -q pytest && python -m pytest tests/test_embedding.py -q

## 8. Copy the artifacts back to Drive

Then on your machine: download these three files into the repo's `artifacts/`, run
`pytest -q` locally, and commit them.

In [ ]:
OUT = "/content/drive/MyDrive/redrob_artifacts"
os.makedirs(OUT, exist_ok=True)
for name in ("candidate_embeddings.npy", "candidate_ids.npy", "embeddings_meta.json"):
    !cp artifacts/$name "$OUT"/
!ls -lh "$OUT"
!cat artifacts/embeddings_meta.json